# 🎯 Python Decorators: A Complete Guide

**Advanced Python Concepts for Interview Preparation**

- **Created:** February 27, 2026
- **Focus:** First-Class Functions, Syntactic Sugar, & Wrapper Patterns
- **Tags:** `python` | `functions` | `decorators` | `interview-prep`

## 📖 The Core Concept in Plain Language

A **decorator** is simply a **wrapper**. It allows you to execute code *before* and *after* a function without modifying the inner logic of that function.

### Why This Matters

Because Python treats functions as **First-Class Objects** (they live in memory just like integers or strings), you can:
- Pass functions as arguments
- Store functions in data structures
- Return functions from other functions
- Reassign function references

The `@decorator_name` syntax is purely **Syntactic Sugar**. Behind the scenes:

```python
my_func = decorator_name(my_func)
```

## 🔍 First-Class Objects in Action

In [ ]:
# Demonstrate: Functions as First-Class Objects

def greet(name):
    return f"Hello, {name}!"

# 1️⃣ Assign to a variable
func_ref = greet
print(f"1. Assign to variable: {func_ref('Alice')}")

# 2️⃣ Store in a data structure
functions = [greet, lambda x: f"Hi, {x}!"]
print(f"2. Store in list: {functions[0]('Bob')}")

# 3️⃣ Pass as an argument
def apply_function(func, arg):
    return func(arg)

print(f"3. Pass as argument: {apply_function(greet, 'Charlie')}")

# 4️⃣ Return from a function
def get_function():
    return greet

func = get_function()
print(f"4. Return from function: {func('Diana')}")

## 🏗️ The Anatomy of a Decorator

A **Higher-Order Function** that takes a function as input and returns an enhanced version.

### Structure:
```python
def decorator(func):           # ← Takes a function as argument
    def wrapper(*args, **kwargs):  # ← Inner function (the wrapper)
        # Setup (before)
        result = func(*args, **kwargs)
        # Teardown (after)
        return result
    return wrapper             # ← Return the wrapper, not the original
```

In [ ]:
import time
from functools import wraps

# Simple Timer Decorator
def timer_decorator(func):
    @wraps(func)  # Preserves original function metadata
    def wrapper(*args, **kwargs):
        # ⏱️ SETUP (before)
        start = time.time()
        print(f"⏳ Starting: {func.__name__}")
        
        # 🎯 EXECUTE original function
        result = func(*args, **kwargs)
        
        # ⏱️ TEARDOWN (after)
        elapsed = time.time() - start
        print(f"✅ Completed in {elapsed:.4f} seconds")
        
        return result
    return wrapper

# Apply the decorator
@timer_decorator
def slow_function():
    """A function that takes some time."""
    time.sleep(0.5)
    return "Done!"

print(slow_function())

## 🔄 Manual Application vs @ Syntax

In [ ]:
def log_decorator(func):
    @wraps(func)
    def wrapper(*args, **kwargs):
        print(f"📝 Calling: {func.__name__}({args}, {kwargs})")
        return func(*args, **kwargs)
    return wrapper

# Approach 1: Manual assignment (no @ syntax)
def add_manual(a, b):
    return a + b

add_manual = log_decorator(add_manual)
print("\n✨ Manual approach:")
result1 = add_manual(5, 3)
print(f"Result: {result1}\n")

# Approach 2: @ Syntax (Syntactic Sugar)
@log_decorator
def add_decorated(a, b):
    return a + b

print("✨ @ Syntax approach:")
result2 = add_decorated(5, 3)
print(f"Result: {result2}")

print(f"\n🔍 Both approaches are identical: {result1 == result2}")

## 🌍 Real-World Example: Retry Decorator

**The Citi Story:** Thousands of endpoints timeout or drop packets. Instead of cluttering every function with `try/catch` loops, abstract the retry logic into a decorator.

### Without Decorator (Messy Boilerplate)
```python
def fetch_from_db():
    for attempt in range(3):
        try:
            # ... complex business logic ...
            return data
        except ConnectionError:
            time.sleep(2 ** attempt)
    raise Exception("Failed after 3 attempts")

# Repeat this boilerplate 50 times... 😩
```

### With Decorator (Clean & Maintainable)
```python
@retry_on_failure(attempts=3, backoff=2)
def fetch_from_db():
    # Pure business logic only
    return data
```

In [ ]:
import random

def retry_on_failure(attempts=3, backoff=1):
    """Decorator factory: returns a decorator with configurable attempts."""
    def decorator(func):
        @wraps(func)
        def wrapper(*args, **kwargs):
            for attempt in range(1, attempts + 1):
                try:
                    print(f"🔄 Attempt {attempt}/{attempts}...")
                    result = func(*args, **kwargs)
                    print(f"✅ Success on attempt {attempt}!")
                    return result
                except Exception as e:
                    if attempt == attempts:
                        print(f"❌ Failed after {attempts} attempts: {e}")
                        raise
                    wait_time = backoff ** attempt
                    print(f"⏳ Waiting {wait_time}s before retry...")
                    time.sleep(wait_time)
        return wrapper
    return decorator

# Simulated flaky database connection
call_count = 0

@retry_on_failure(attempts=3, backoff=0.5)  # backoff in 0.5s for demo
def query_database():
    global call_count
    call_count += 1
    if call_count < 3:
        raise ConnectionError("Network timeout")
    return "✨ Database result: [1, 2, 3, 4, 5]"

print(query_database())

## 🎭 Stacking Multiple Decorators

In [ ]:
def auth_required(func):
    @wraps(func)
    def wrapper(*args, **kwargs):
        print("🔐 Checking authentication...")
        return func(*args, **kwargs)
    return wrapper

def rate_limit(func):
    @wraps(func)
    def wrapper(*args, **kwargs):
        print("⚡ Checking rate limit...")
        return func(*args, **kwargs)
    return wrapper

# Stack decorators: executed BOTTOM-UP
@auth_required      # Applied SECOND
@rate_limit         # Applied FIRST
def api_endpoint():
    print("📡 Processing API request...")
    return {"status": "success", "data": [1, 2, 3]}

result = api_endpoint()
print(f"Result: {result}")

## 🎤 5 Interview Q&A

### Q1: What does it mean that Python functions are "First-Class Citizens"?

**Answer:** Functions are full objects in memory. They can be assigned to variables, stored in data structures (lists, dicts), passed as arguments, returned by other functions, and modified at runtime. This is the foundation that makes decorators possible.

---

### Q2: What is the anatomy of a Decorator?

**Answer:** A decorator is a **Higher-Order Function** that takes a function as input and returns an enhanced version. It:
1. Takes `func` as an argument
2. Defines an inner `wrapper()` function
3. The `wrapper()` does setup, calls `func()`, does teardown
4. Returns the `wrapper` object (not the original)

---

### Q3: How do you apply a decorator without the `@` syntax?

**Answer:** Manual reassignment. If `my_timer` is a decorator and `say_hello` is a function:
```python
say_hello = my_timer(say_hello)
```
The `@decorator` syntax is syntactic sugar that does exactly this under the hood.

---

### Q4: Why use a decorator instead of adding code inside the function?

**Answer:** **Separation of Concerns**. Decorators keep business logic pure and separate from cross-cutting concerns. If you have 50 API endpoints needing authentication, a single `@requires_auth` decorator is cleaner than copy/pasting auth logic into all 50 functions. It's maintainability, testability, and DRY (Don't Repeat Yourself).

---

### Q5: What are common real-world use cases for decorators?

**Answer:**
- **Timing:** `@timer` — measure function execution time
- **Authorization:** `@login_required` — check user permissions
- **Retries:** `@retry(attempts=3)` — handle flaky connections
- **Caching:** `@lru_cache` — memoize expensive computations
- **Logging:** `@log_calls` — audit function invocations
- **Rate Limiting:** `@rate_limit(calls=10/min)` — throttle API usage

## 📚 Key Terminology to Master

| Term | Definition | Example |
|------|-----------|----------|
| **First-Class Objects/Citizens** | Functions can be assigned, passed, stored, returned like any other value | `func = my_function` |
| **Syntactic Sugar** | Convenient syntax that reduces verbosity without changing functionality | `@decorator` instead of `func = decorator(func)` |
| **Higher-Order Functions** | Functions that take functions as arguments or return functions | `decorator(func)` or `map(func, list)` |
| **Separation of Concerns** | Each component has one responsibility; cross-cutting concerns (logging, auth) separated from core logic | `@auth` on API endpoint |
| **Boilerplate Abstraction** | Moving repetitive code patterns into reusable decorators | `@retry` vs copy/paste try/catch in 50 functions |
| **Closure** | Inner function remembers variables from enclosing scope | `wrapper()` accesses `func` from decorator's scope |
| **Decorator Factory** | A function that returns a decorator (allows parameterization) | `@retry(attempts=3)` |
| **`@wraps`** | Preserves original function's metadata (name, docstring) | `from functools import wraps` |

## 💼 The Citi Narrative: From Chaos to Elegance

### The Problem

At Citi, we deal with **thousands of downstream endpoints** and **database connections** that occasionally timeout or drop packets. Early on, engineers were cluttering their core calculation functions with massive `try/catch` and `time.sleep()` loops just to safely retry failures.

### The Solution

We cleaned up the entire codebase by **abstracting that boilerplate into custom Python decorators**. Leveraging functions as **first-class objects**, we allowed developers to:

1. **Write pure business logic** without error handling clutter
2. **Stick an `@` decorator on top** of it
3. **Let the wrapper handle resilience and telemetry**

### The Code Pattern

```python
@retry_on_failure(attempts=3)
@timer
@log_calls
def calculate_risk_metric(portfolio_id):
    # Pure business logic — no error handling noise
    return query_db(portfolio_id).compute_var()
```

### The Impact

- **Code cleanliness:** Business logic separated from operational concerns
- **Maintainability:** Change retry logic once; it applies everywhere
- **Testability:** Mock decorators independently; test core logic in isolation
- **Scalability:** New patterns (e.g., circuit breaker, metrics) added as new decorators

## 💪 Practice Exercises

In [ ]:
# EXERCISE 1: Write a Decorator that Converts Output to Uppercase

def uppercase_output(func):
    @wraps(func)
    def wrapper(*args, **kwargs):
        result = func(*args, **kwargs)
        if isinstance(result, str):
            return result.upper()
        return result
    return wrapper

@uppercase_output
def greet(name):
    return f"hello, {name}!"

print(f"Exercise 1: {greet('alice')}")
# Expected: "HELLO, ALICE!"

In [ ]:
# EXERCISE 2: Write a Parameterized Decorator that Repeats Function Output

def repeat_output(times=2):
    def decorator(func):
        @wraps(func)
        def wrapper(*args, **kwargs):
            result = func(*args, **kwargs)
            return result * times  # Works for strings and numbers
        return wrapper
    return decorator

@repeat_output(times=3)
def get_message():
    return "Python "

print(f"Exercise 2: {get_message()}")
# Expected: "Python Python Python "

In [ ]:
# EXERCISE 3: Write a Decorator that Catches and Logs Exceptions

def safe_execution(func):
    @wraps(func)
    def wrapper(*args, **kwargs):
        try:
            return func(*args, **kwargs)
        except Exception as e:
            print(f"⚠️  Exception in {func.__name__}: {type(e).__name__}: {e}")
            return None
    return wrapper

@safe_execution
def divide(a, b):
    return a / b

print(f"Exercise 3a (valid): {divide(10, 2)}")
print(f"Exercise 3b (error): {divide(10, 0)}")
# Expected: "Exercise 3a (valid): 5.0"
# Expected: "⚠️  Exception in divide: ZeroDivisionError: division by zero"
# Expected: "Exercise 3b (error): None"

## 🎯 Summary: Key Takeaways

### What You've Learned

1. ✅ **First-Class Objects:** Functions are objects that can be assigned, passed, stored, and returned
2. ✅ **Decorator Anatomy:** A wrapper function inside a function that takes a function as input
3. ✅ **Syntactic Sugar:** `@decorator` is shorthand for `func = decorator(func)`
4. ✅ **Higher-Order Functions:** Functions that operate on other functions
5. ✅ **Separation of Concerns:** Keep business logic pure; abstract cross-cutting concerns into decorators
6. ✅ **Parameterized Decorators:** Use decorator factories (`@decorator(param=value)`) for flexibility
7. ✅ **Real-World Patterns:** Retry logic, authentication, timing, logging, rate limiting

### When to Use Decorators

- ✅ **DO** use decorators when you need the same behavior across many functions
- ✅ **DO** use decorators to keep business logic separate from operational concerns
- ✅ **DO** use decorators for debugging, monitoring, and validation
- ❌ **DON'T** overuse decorators in ways that make code hard to follow
- ❌ **DON'T** nest more than 2-3 decorators without documentation

### Interview Confidence Checklist

- [ ] Can explain what First-Class Objects means
- [ ] Can write a basic decorator from scratch
- [ ] Can explain `@decorator` vs manual assignment
- [ ] Can write a decorator factory with parameters
- [ ] Can apply multiple decorators and explain execution order
- [ ] Can name 5+ real-world use cases
- [ ] Can explain why `from functools import wraps` matters

---

**"Simplicity and clarity is Gold."** — Sean's Study Mantra

Master decorators, and you've mastered one of Python's most powerful and elegant patterns. 🚀